# Generación de BoSE para el dataset de depresión

Usa los artefactos ya entrenados sobre Dreaddit:
- `word_to_subemotion.json` → lookup palabra → subemoción
- `MODELOS/best_model_bose.pkl` → TF-IDF ya ajustado

**Problema:** `.toarray()` sobre 77k×182k requiere ~106 GB → MemoryError.  
**Solución:** guardar en formato **sparse** (`.npz`) y hacer la inferencia en **chunks** para no materializar nunca la matriz densa completa.

## 1. Imports

In [1]:
import json
import joblib
import numpy as np
import pandas as pd
import scipy.sparse as sp
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader

from bert_wrapper_vanila import BertWithFeatures, CustomDataset

## 2. Cargar textos de depresión

In [3]:
df = pd.read_csv("csv del analisis/depresion por posts/df_nivelPosts_etiquetado.csv")
df["text"] = df["text"].fillna("").astype(str)

print(f"Posts: {len(df)}")
print(f"Usuarios únicos: {df['id_subject'].nunique()}")
print(f"Distribución label_depresion:\n{df['label_depresion'].value_counts().sort_index()}")

Posts: 77681
Usuarios únicos: 168
Distribución label_depresion:
label_depresion
0    23501
1    11735
2    20172
3    22273
Name: count, dtype: int64


## 3. Cargar diccionario y enmascarar textos

In [4]:
with open("word_to_subemotion.json", "r") as f:
    word_subemotion = json.load(f)

print(f"Palabras en diccionario: {len(word_subemotion)}")

def mask_text(text):
    return " ".join(word_subemotion.get(w, "UNK") for w in text.lower().split())

print("Enmascarando textos...")
texts_masked = [mask_text(t) for t in df["text"]]
print(f"Hecho. Ejemplo: {texts_masked[0][:80]}")

Palabras en diccionario: 23461
Enmascarando textos...
Hecho. Ejemplo: positive_6 joy_96 disgust_38 trust_214 surprise_12 negative_220 anger_53 sadness


## 4. Transformar con TF-IDF → guardar en sparse

La matriz sparse ocupa ~pocos MB en disco (solo guarda los valores no cero).  
**Nunca llamamos `.toarray()`** sobre la matriz completa.

In [5]:
pipeline = joblib.load("MODELOS/best_model_bose.pkl")
tfidf = pipeline.named_steps["tfidf"]

print(f"Vocabulario TF-IDF: {len(tfidf.vocabulary_)} términos")

print("Transformando (formato sparse, sin .toarray())...")
X_bose_sparse = tfidf.transform(texts_masked)  # scipy sparse, cabe en RAM
print(f"Shape: {X_bose_sparse.shape}")
print(f"Elementos no cero: {X_bose_sparse.nnz:,}  ({100*X_bose_sparse.nnz/np.prod(X_bose_sparse.shape):.3f}% denso)")

# Guardar en formato sparse
sp.save_npz("bose_depression_sparse.npz", X_bose_sparse)
print("Guardado: bose_depression_sparse.npz")

Vocabulario TF-IDF: 182528 términos
Transformando (formato sparse, sin .toarray())...
Shape: (77681, 182528)
Elementos no cero: 4,527,685  (0.032% denso)
Guardado: bose_depression_sparse.npz
